# 08 — Analisis Korelasi Faktor Kontekstual

Uji statistik korelasi antara fitur kontekstual (`match_outcome_for_player`, `duration`, `tier`, `phase`, `period`) dengan label sentimen dan toksisitas, dengan koreksi Benjamini-Hochberg FDR (q=0.05).

**Pra-syarat**: notebook 05 selesai.

**Output**: `reports/correlation_tests.csv`, `reports/correlation_summary.md`, plot `reports/plots/correlation_*.{png,svg}`.

In [1]:
# Sel 1: Setup
import sys
from pathlib import Path
_here = Path.cwd()
_root = next((p for p in [_here, *_here.parents] if (p / 'src').is_dir() and (p / 'configs').is_dir()), _here)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
import os
os.chdir(_root)

from src.runtime import load_config, print_banner, RunLog
config = load_config('configs/experiment.yaml')
print_banner('08_correlation_analysis', config)
run_log = RunLog(notebook='08_correlation_analysis', config_path='configs/experiment.yaml')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROCESSED_ROOT = Path(config['data']['processed_root'])
INF_ROOT = Path(config['data']['inference_root'])
REPORTS = Path('reports')
PLOTS = REPORTS / 'plots'
PLOTS.mkdir(parents=True, exist_ok=True)

SENT_LABELS = config['labels']['sentiment_classes']
TOX_LABELS = config['labels']['toxicity_labels']

c:\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Notebook: 08_correlation_analysis
Experiment: thesis-sentiment-toxicity-dota2-decade
Seed: 42  |  Git: be03038
Started at: 2026-05-05T07:28:57+00:00Z
Versi paket:
  - python: 3.10.0
  - transformers: 5.5.0
  - torch: 2.8.0+cu128
  - datasets: 4.8.5
  - scikit-learn: 1.7.1
  - pandas: 2.3.3
  - numpy: 1.26.4


In [2]:
# Sel 2: Load merged data
from src.analysis.temporal import merge_inference_with_processed
BEST_SENT = 'roberta'
BEST_TOX = 'detoxify'
df = merge_inference_with_processed(
    processed_root=PROCESSED_ROOT,
    sentiment_inference_root=INF_ROOT / f'{BEST_SENT}_sentiment',
    toxicity_inference_root=INF_ROOT / f'{BEST_TOX}_toxicity',
    sentiment_labels=SENT_LABELS,
    toxicity_labels=TOX_LABELS,
)
# Tambah kolom turunan period
df['period'] = pd.cut(
    df['year'].astype('Int64'),
    bins=[2015, 2018, 2021, 2026],
    labels=['2016-2018', '2019-2021', '2022-2026'],
    right=True,
).astype('string')
df['duration_bin'] = pd.cut(
    df['duration'].astype('Int64'),
    bins=[0, 1800, 2700, 7200],
    labels=['short', 'medium', 'long'],
).astype('string')

# Sentiment label string from predicted_label
if 'predicted_label' in df.columns:
    df['sentiment_label'] = df['predicted_label'].astype('string')

print(f'Total rows: {len(df):,}')
print(df[['period', 'tier', 'phase', 'match_outcome_for_player', 'duration_bin']].apply(lambda c: c.value_counts(dropna=False).head(5)))

Total rows: 1,380,867
             period     tier    phase  match_outcome_for_player  duration_bin
2016-2018    271746     <NA>     <NA>                      <NA>          <NA>
2019-2021    426392     <NA>     <NA>                      <NA>          <NA>
2022-2026    682729     <NA>     <NA>                      <NA>          <NA>
DPC Tour       <NA>    56266     <NA>                      <NA>          <NA>
Lainnya        <NA>  1266476     <NA>                      <NA>          <NA>
Major          <NA>    34106     <NA>                      <NA>          <NA>
TI             <NA>    24019     <NA>                      <NA>          <NA>
grand_final    <NA>     <NA>      880                      <NA>          <NA>
lainnya        <NA>     <NA>  1375086                      <NA>          <NA>
long           <NA>     <NA>     <NA>                      <NA>        248170
loss           <NA>     <NA>     <NA>                    717410          <NA>
medium         <NA>     <NA>     <NA>     

In [3]:
# Sel 3: Suite uji korelasi
from src.analysis.correlation import (
    chi_square_test, mannwhitney_test, spearman_test, kruskal_test
)

# Filter valid rows untuk uji yang membutuhkan outcome
valid = df[df['match_outcome_for_player'].isin(['win', 'loss'])].copy()

rows = []
# 1. outcome × sentiment (chi-square)
if 'sentiment_label' in valid.columns:
    r = chi_square_test(valid, 'match_outcome_for_player', 'sentiment_label')
    r.update({'feature': 'match_outcome_for_player', 'label': 'sentiment_label'}); rows.append(r)
# 2. outcome × max_toxicity_prob (Mann-Whitney)
if 'max_toxicity_prob' in valid.columns:
    r = mannwhitney_test(valid, 'match_outcome_for_player', 'max_toxicity_prob')
    r.update({'feature': 'match_outcome_for_player', 'label': 'max_toxicity_prob'}); rows.append(r)
# 3. duration × sentiment_score (Spearman)
if 'sentiment_score' in df.columns:
    r = spearman_test(df, 'duration', 'sentiment_score')
    r.update({'feature': 'duration', 'label': 'sentiment_score'}); rows.append(r)
# 4. duration × max_toxicity_prob (Spearman)
if 'max_toxicity_prob' in df.columns:
    r = spearman_test(df, 'duration', 'max_toxicity_prob')
    r.update({'feature': 'duration', 'label': 'max_toxicity_prob'}); rows.append(r)
# 5. tier × max_toxicity_prob (Kruskal-Wallis)
if 'max_toxicity_prob' in df.columns:
    r = kruskal_test(df, 'tier', 'max_toxicity_prob')
    r.update({'feature': 'tier', 'label': 'max_toxicity_prob'}); rows.append(r)
# 6. tier × sentiment (chi-square)
if 'sentiment_label' in df.columns:
    r = chi_square_test(df, 'tier', 'sentiment_label')
    r.update({'feature': 'tier', 'label': 'sentiment_label'}); rows.append(r)
# 7. phase × max_toxicity_prob
if 'max_toxicity_prob' in df.columns:
    r = kruskal_test(df, 'phase', 'max_toxicity_prob')
    r.update({'feature': 'phase', 'label': 'max_toxicity_prob'}); rows.append(r)
# 8. phase × sentiment
if 'sentiment_label' in df.columns:
    r = chi_square_test(df, 'phase', 'sentiment_label')
    r.update({'feature': 'phase', 'label': 'sentiment_label'}); rows.append(r)
# 9. period × max_toxicity_prob
if 'max_toxicity_prob' in df.columns:
    r = kruskal_test(df, 'period', 'max_toxicity_prob')
    r.update({'feature': 'period', 'label': 'max_toxicity_prob'}); rows.append(r)
# 10. period × sentiment
if 'sentiment_label' in df.columns:
    r = chi_square_test(df, 'period', 'sentiment_label')
    r.update({'feature': 'period', 'label': 'sentiment_label'}); rows.append(r)

tests_df = pd.DataFrame(rows)
print(tests_df[['feature', 'label', 'test', 'test_statistic', 'p_value', 'effect_size', 'n']].to_string(index=False))

                 feature             label           test  test_statistic       p_value  effect_size       n
match_outcome_for_player   sentiment_label     chi-square    3.415839e+02  6.698957e-75     0.015876 1355212
match_outcome_for_player max_toxicity_prob   mann-whitney    2.151164e+11  0.000000e+00     0.059735 1355212
                duration   sentiment_score       spearman   -5.494864e-02  0.000000e+00    -0.054949 1380867
                duration max_toxicity_prob       spearman   -8.048506e-02  0.000000e+00    -0.080485 1380867
                    tier max_toxicity_prob kruskal-wallis    8.874146e+02 4.751864e-192     0.000640 1380867
                    tier   sentiment_label     chi-square    6.526810e+01  3.802982e-12     0.004861 1380867
                   phase max_toxicity_prob kruskal-wallis    4.043782e+01  1.655919e-09     0.000028 1380867
                   phase   sentiment_label     chi-square    2.850630e+01  9.846803e-06     0.003213 1380867
                  p

In [4]:
# Sel 4: BH-FDR koreksi
from src.analysis.multiple_testing import apply_to_dataframe
tests_df = apply_to_dataframe(tests_df, p_col='p_value', q=float(config['multiple_testing']['q']))
tests_df.to_csv(REPORTS / 'correlation_tests.csv', index=False)
print(tests_df[['feature', 'label', 'test', 'p_value', 'p_adj_bh', 'significant_after_bh']].to_string(index=False))
run_log.add_output(REPORTS / 'correlation_tests.csv')

                 feature             label           test       p_value      p_adj_bh  significant_after_bh
match_outcome_for_player   sentiment_label     chi-square  6.698957e-75  9.569939e-75                  True
match_outcome_for_player max_toxicity_prob   mann-whitney  0.000000e+00  0.000000e+00                  True
                duration   sentiment_score       spearman  0.000000e+00  0.000000e+00                  True
                duration max_toxicity_prob       spearman  0.000000e+00  0.000000e+00                  True
                    tier max_toxicity_prob kruskal-wallis 4.751864e-192 7.919774e-192                  True
                    tier   sentiment_label     chi-square  3.802982e-12  4.753727e-12                  True
                   phase max_toxicity_prob kruskal-wallis  1.655919e-09  1.839910e-09                  True
                   phase   sentiment_label     chi-square  9.846803e-06  9.846803e-06                  True
                  period max

In [5]:
# Sel 5: Plot pendukung
if 'sentiment_label' in df.columns:
    fig, ax = plt.subplots(figsize=(7, 4))
    ct = pd.crosstab(valid['match_outcome_for_player'], valid['sentiment_label'], normalize='index')
    ct.plot(kind='bar', stacked=True, ax=ax)
    ax.set_title('Sentimen × Hasil Pertandingan'); ax.set_ylabel('Proporsi'); ax.legend(title='sentiment')
    plt.tight_layout(); plt.savefig(PLOTS / 'correlation_outcome_sentiment.png', dpi=120); plt.savefig(PLOTS / 'correlation_outcome_sentiment.svg'); plt.close(fig)

if 'max_toxicity_prob' in df.columns:
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.violinplot(data=valid, x='match_outcome_for_player', y='max_toxicity_prob', ax=ax)
    ax.set_title('Toksisitas (max prob) × Hasil')
    plt.tight_layout(); plt.savefig(PLOTS / 'correlation_outcome_toxicity.png', dpi=120); plt.savefig(PLOTS / 'correlation_outcome_toxicity.svg'); plt.close(fig)

    fig, ax = plt.subplots(figsize=(8, 4))
    sns.boxplot(data=df.dropna(subset=['tier']), x='tier', y='max_toxicity_prob', order=['TI', 'Major', 'DPC Tour', 'Lainnya'], ax=ax)
    ax.set_title('Toksisitas × Tier Turnamen')
    plt.tight_layout(); plt.savefig(PLOTS / 'correlation_tier_toxicity.png', dpi=120); plt.savefig(PLOTS / 'correlation_tier_toxicity.svg'); plt.close(fig)

    # Phase
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.boxplot(data=df.dropna(subset=['phase']), x='phase', y='max_toxicity_prob', ax=ax)
    ax.set_title('Toksisitas × Fase Turnamen')
    plt.tight_layout(); plt.savefig(PLOTS / 'correlation_phase_toxicity.png', dpi=120); plt.savefig(PLOTS / 'correlation_phase_toxicity.svg'); plt.close(fig)

    # Duration scatter (sample untuk visualisasi)
    sample = df.sample(min(50_000, len(df)), random_state=42)
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.regplot(data=sample, x='duration', y='max_toxicity_prob', lowess=True, scatter_kws={'alpha': 0.05, 's': 4}, ax=ax)
    ax.set_title('Toksisitas × Durasi Pertandingan (LOWESS)')
    plt.tight_layout(); plt.savefig(PLOTS / 'correlation_duration_toxicity.png', dpi=120); plt.savefig(PLOTS / 'correlation_duration_toxicity.svg'); plt.close(fig)

for f in PLOTS.glob('correlation_*.png'):
    run_log.add_output(f)

In [6]:
# Sel 6: Robustness — agregat per match_id sebagai kontrol clustering effect
agg_per_match = df.groupby('match_id').agg(
    mean_toxicity=('max_toxicity_prob', 'mean'),
    mean_sentiment_score=('sentiment_score', 'mean') if 'sentiment_score' in df.columns else ('match_id', 'count'),
    duration=('duration', 'first'),
    tier=('tier', 'first'),
    phase=('phase', 'first'),
    n_messages=('match_id', 'count'),
).reset_index()

from src.analysis.correlation import spearman_test, kruskal_test
robust_rows = []
if 'duration' in agg_per_match.columns and 'mean_toxicity' in agg_per_match.columns:
    r = spearman_test(agg_per_match, 'duration', 'mean_toxicity')
    r.update({'feature': 'duration', 'label': 'mean_toxicity', 'level': 'per_match'}); robust_rows.append(r)
if 'tier' in agg_per_match.columns:
    r = kruskal_test(agg_per_match, 'tier', 'mean_toxicity')
    r.update({'feature': 'tier', 'label': 'mean_toxicity', 'level': 'per_match'}); robust_rows.append(r)
robust_df = pd.DataFrame(robust_rows)
if not robust_df.empty:
    robust_df.to_csv(REPORTS / 'correlation_robustness_per_match.csv', index=False)
    run_log.add_output(REPORTS / 'correlation_robustness_per_match.csv')
    print(robust_df.to_string(index=False))

          test  test_statistic  p_value  effect_size      n interpretation  feature         label     level  df
      spearman       -0.147458 0.000000    -0.147458 197695    significant duration mean_toxicity per_match NaN
kruskal-wallis        1.764172 0.622762    -0.000006 197695             ns     tier mean_toxicity per_match 3.0


In [7]:
# Sel 7: correlation_summary.md
lines = ['# Ringkasan Korelasi Fitur Kontekstual\n', '## Tabel Hasil Uji\n']
cols = ['feature', 'label', 'test', 'test_statistic', 'p_value', 'p_adj_bh', 'significant_after_bh', 'effect_size', 'n']
cols = [c for c in cols if c in tests_df.columns]
lines.append(tests_df[cols].to_markdown(index=False, floatfmt='.4f'))

sig = tests_df[tests_df['significant_after_bh']].sort_values('effect_size', key=lambda c: c.abs(), ascending=False)
lines.append('\n## Finding Signifikan setelah BH-FDR (urut effect size desc)\n')
if not sig.empty:
    for _, r in sig.iterrows():
        lines.append(f'- `{r.feature}` × `{r.label}` ({r.test}): stat={r.test_statistic:.3f}, p_adj={r.p_adj_bh:.4f}, effect={r.effect_size:.3f}, n={int(r.n)}')
else:
    lines.append('_Tidak ada finding signifikan setelah koreksi BH-FDR._')

lines += [
    '\n## Catatan Limitasi\n',
    '- Pesan dalam satu pertandingan TIDAK independen (clustering effect). Analisis robustness pada level per-pertandingan dilaporkan di `correlation_robustness_per_match.csv`.',
    '- Kolom `phase` mayoritas bernilai `lainnya` karena dataset Dota 2 publik tidak menyertakan fase per-match secara eksplisit (lihat `reports/contextual_features.md` §4).',
    '- Effect size yang sangat kecil (mis. Cramér\'s V < 0.1, |rho| < 0.1) menunjukkan signifikansi statistik tanpa relevansi praktis pada n besar.',
]
(REPORTS / 'correlation_summary.md').write_text('\n'.join(lines), encoding='utf-8')
print(f'Tertulis: {REPORTS / "correlation_summary.md"}')
run_log.add_output(REPORTS / 'correlation_summary.md')
run_log.save('reports/run_log.csv')

Tertulis: reports\correlation_summary.md
[run_log] 08_correlation_analysis → 51.18s, 8 outputs, 0 warnings → reports\run_log.csv
